# Neon-Pulse gesture trainer
Upload `dataset.json` from `collect.html`, run all cells, download `gesture.pt`, put it at `vision_service/weights/gesture.pt`.

In [ ]:
!pip -q install opencv-python-headlessimport base64, json, cv2, numpy as np, torch, torch.nn as nnfrom google.colab import filesup = files.upload(); d = json.load(open(list(up)[0]))LABELS = ['idle','tap','fist']

In [ ]:
class GestureCNN(nn.Module):    def __init__(self):        super().__init__()        blk = lambda i, o: nn.Sequential(nn.Conv2d(i, o, 3, padding=1), nn.BatchNorm2d(o), nn.ReLU(), nn.MaxPool2d(2))        self.f = nn.Sequential(blk(1,16), blk(16,32), blk(32,64)); self.h = nn.Sequential(nn.Flatten(), nn.Dropout(.3), nn.Linear(64*8*8, 3))    def forward(self, x): return self.h(self.f(x))

In [ ]:
X, y = [], []for i, l in enumerate(LABELS):    for u in d.get(l, []):        im = cv2.imdecode(np.frombuffer(base64.b64decode(u.split(',')[1]), np.uint8), cv2.IMREAD_GRAYSCALE)        if im is not None: X.append(cv2.resize(im, (64,64))/255.0); y.append(i)X = torch.tensor(np.array(X), dtype=torch.float32)[:, None]; y = torch.tensor(y); print(X.shape, torch.bincount(y))

In [ ]:
p = torch.randperm(len(X)); cut = int(len(X)*.8); tr, va = p[:cut], p[cut:]m = GestureCNN(); opt = torch.optim.Adam(m.parameters(), 1e-3); lf = nn.CrossEntropyLoss()for ep in range(30):    m.train(); tr = tr[torch.randperm(len(tr))]    for i in range(0, len(tr), 32):        idx = tr[i:i+32]; b = X[idx]*(0.8+0.4*torch.rand(len(idx),1,1,1)); opt.zero_grad(); lf(m(b.clamp(0,1)), y[idx]).backward(); opt.step()    m.eval()    with torch.no_grad(): print(ep+1, 'val_acc', (m(X[va]).argmax(1)==y[va]).float().mean().item())torch.save(m.state_dict(), 'gesture.pt'); files.download('gesture.pt')